In [15]:
import numpy as np
from deepcave import Recorder, Objective
from deepcave.runs.converters.deepcave import DeepCAVERun
from deepcave.plugins.summary.footprint import FootPrint
from deepcave.plugins.hyperparameter.importances import Importances
import io 
from PIL import Image
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as plt
from arlbench.core.algorithms import DQN, PPO, SAC
from pathlib import Path
from ConfigSpace import ConfigurationSpace, Float, Categorical
import math
import os

In [36]:
replacement_dict = {"atari_qbert": "atari", 'atari_double_dunk': "atari", 'atari_phoenix': "atari", 'atari_this_game': "atari", 
                    'box2d_lunar_lander': "box2d", "box2d_bipedal_walker": "box2d", 'cc_acrobot': "cc", 'cc_cartpole': "cc", 'cc_mountain_car': "cc", "cc_pendulum": "cc", 'cc_continuous_mountain_car': "cc",
                    'minigrid_door_key': "minigrid", 'minigrid_empty_random': "minigrid", 'minigrid_four_rooms': "minigrid", 
                    'minigrid_unlock': "minigrid", "brax_ant": "brax", "brax_halfcheetah": "brax", "brax_hopper": "brax", "brax_humanoid": "brax"}
algorithms = {"dqn": DQN, "ppo": PPO, "sac": SAC}
envs = ["atari_qbert", 'atari_double_dunk', 'atari_phoenix', 'atari_this_game', 
        'box2d_lunar_lander', 'box2d_bipedal_walker', 'cc_acrobot', 'cc_cartpole', 'cc_mountain_car', 'cc_pendulum', 'cc_continuous_mountain_car',
        'minigrid_door_key', 'minigrid_empty_random', 'minigrid_four_rooms', 'minigrid_unlock', 'brax_ant', 'brax_halfcheetah', 'brax_hopper', 'brax_humanoid']


In [17]:
def plotly_fig2array(fig):
    #convert Plotly fig to  an array
    fig_bytes = fig.to_image(format="png")
    buf = io.BytesIO(fig_bytes)
    img = Image.open(buf)
    return np.asarray(img)

In [33]:
def data_to_deepcave(data, algorithm, domain=False, save_path=None):
    if domain in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain]
        data = data[data["Environment"].isin(subdomains)]
    else:
        data = data[data["Environment"].isin([domain])]
    default_configspace = algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names()
    configspace = ConfigurationSpace()
    for c in default_configspace:
        if c in ["buffer_prio_sampling", "use_target_network", "alpha_auto", "normalize_advantage"]:
            hp = Categorical(c, [True, False])
            configspace.add_hyperparameter(hp)
        else:
            hp = Float(c, bounds=(0, 1e7))
            configspace.add_hyperparameter(hp)
    default = configspace.get_default_configuration()

    target_path = f"../runscripts/configs/initial_design/{domain}_{algorithm}.csv"
    try:
        all_configs = pd.read_csv(target_path)
    except:
        print(f"Could not read {target_path}")
        return
    accuracy = Objective("reward_mean", lower=min(all_configs["performance"]), upper=max(all_configs["performance"]), optimize="upper")
    save_path = save_path if save_path else f"deepcave_logs/{algorithm}_{domain}"

    with Recorder(configspace, objectives=[accuracy], save_path=save_path) as r:
        for index, d in data.iterrows():
            current_hps = all_configs.iloc[d["Configuration"]]
            config = default
            for c in current_hps.keys():
                key = c.split(".")[-1]
                if key in config.keys():
                    if math.isnan(current_hps[c]):
                        current_hps[c] = 0

                    try:
                        value = float(current_hps[c])
                    except:
                        current_hps[c]
                    config[key] = value
            r.start(config, 1)
            r.end(costs=current_hps["performance"], seed=d["Seed"])
    return save_path

In [30]:
def get_importances(path, algorithm):
    # Instantiate the run
    run = DeepCAVERun.from_path(Path(path))
    objective_id = run.get_objective_ids()[0]
    budget_ids = run.get_budget_ids()

    # Instantiate the plugin
    plugin = Importances()

    inputs = plugin.generate_inputs(
        objective_id=objective_id,
        hyperparameter_names= algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names(), 
        budget_ids=budget_ids,
        method="local", n_hps=10, n_trees=10
    )
    # Note: Filter variables are not considered.
    outputs = plugin.generate_outputs(run, inputs)

    # Finally, you can load the figure. Here, the filter variables play a role.
    # Alternatively: Use the matplotlib output (`load_mpl_outputs`) if available.
    figure = plugin.load_outputs(run, inputs, outputs)  # plotly.go figure
    return figure

In [20]:
def get_footprint(path):
    # Instantiate the run
    run = DeepCAVERun.from_path(Path(path))
    objective_id = run.get_objective_ids()[0]
    budget_id = run.get_budget_ids()[-1]

    # Instantiate the plugin
    plugin = FootPrint()

    inputs = plugin.generate_inputs(
        objective_id=objective_id,
        budget_id=budget_id,
        details=True,
        show_supports=True,
        show_borders=True,
    )
    # Note: Filter variables are not considered.
    outputs = plugin.generate_outputs(run, inputs)

    # Finally, you can load the figure. Here, the filter variables play a role.
    # Alternatively: Use the matplotlib output (`load_mpl_outputs`) if available.
    figure = plugin.load_outputs(run, inputs, outputs)  # plotly.go figure
    return figure

In [21]:
def normalize_env_scores(data):
    groups = data.groupby("Environment")["Score"]
    data["Score"] = (data["Score"] - groups.transform('min')) / (groups.transform('max') - groups.transform('min'))
    return data

In [22]:
def get_boxenplot(data, domain=False, domain_single=None):
    if domain:
        data["Environment"] = data["Environment"].replace(replacement_dict)
    if domain_single in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain_single]
        data = data[data["Environment"].isin(subdomains)]
    fig = sns.boxenplot(data, x="Environment", y="Score")
    plt.xticks(rotation=70)
    plt.tight_layout()
    return fig

def get_kdeplot(data, domain=False, domain_single=None):
    if domain:
        data["Environment"] = data["Environment"].replace(replacement_dict)
    if domain_single in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain_single]
        data = data[data["Environment"].isin(subdomains)]
    return sns.kdeplot(data, x="Score", hue="Environment")

def get_pairplot(data, domain=False, domain_single=None):
    if domain:
        data["Environment"] = data["Environment"].replace(replacement_dict)
    if domain_single in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain_single]
        data = data[data["Environment"].isin(subdomains)]
    return sns.pairplot(data, hue="Environment", kind="kde")

In [23]:
def load_data(algorithm):
    source_file = f"data/{algorithm}/performances.csv"
    meta_file = f"data/{algorithm}/metadata.json"
    with open(meta_file) as f:
        metadata = json.load(f)
    return pd.read_csv(source_file), metadata

In [39]:
algorithm = "dqn"
performance_data, metadata = load_data(algorithm)
performance_data = normalize_env_scores(performance_data)
for env in envs:
   if not os.path.isdir(f"deepcave_logs/{algorithm}_{env}"):
       print(f"Parsing {env}")
       data_path = data_to_deepcave(performance_data, algorithm, domain=env)

Parsing atari_qbert


/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_23465/3786550976.py:35: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Parsing atari_double_dunk


/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_23465/3786550976.py:35: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Parsing atari_phoenix


...

Parsing minigrid_four_rooms


/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_23465/3786550976.py:35: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Parsing minigrid_unlock


/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_23465/3786550976.py:35: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Parsing brax_ant
Could not read ../runscripts/configs/initial_design/brax_ant_dqn.csv
Parsing brax_halfcheetah
Could not read ../runscripts/configs/initial_design/brax_halfcheetah_dqn.csv
Parsing brax_hopper
Could not read ../runscripts/configs/initial_design/brax_hopper_dqn.csv
Parsing brax_humanoid
Could not read ../runscripts/configs/initial_design/brax_humanoid_dqn.csv


In [25]:
get_boxenplot(performance_data.copy(), True)
plt.figure()
get_boxenplot(performance_data.copy())
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="atari")
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="cc")
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="minigrid")
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="box2d")


<Axes: xlabel='Environment', ylabel='Score'>

In [26]:
get_kdeplot(performance_data.copy(), True)
plt.figure()
get_kdeplot(performance_data.copy())
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="atari")
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="cc")
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="minigrid")
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="box2d")

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_23465/3652807460.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()


<Axes: xlabel='Score', ylabel='Density'>

In [27]:
get_pairplot(performance_data.copy(), True)
plt.figure()
get_pairplot(performance_data.copy())
plt.figure()
get_pairplot(performance_data.copy(), domain_single="atari")
plt.figure()
get_pairplot(performance_data.copy(), domain_single="cc")
plt.figure()
get_pairplot(performance_data.copy(), domain_single="minigrid")
plt.figure()
get_pairplot(performance_data.copy(), domain_single="box2d")

In [28]:
data_path = "deepcave_logs/dqn_cc_acrobot"
img = get_footprint(Path(data_path) / "run_1")
img[0].show()

100%|██████████| 2560/2560 [00:14<00:00, 178.51it/s]

deepcave.evaluators.footprint (INFO): Added 2560 configurations.
deepcave.evaluators.footprint (INFO): Starting to calculate distances and add border and random configurations...


deepcave.evaluators.footprint (INFO): Found 2600 configurations...
deepcave.evaluators.footprint (INFO): Found 2700 configurations...
deepcave.evaluators.footprint (INFO): Found 3000 configurations...
deepcave.evaluators.footprint (INFO): Found 3100 configurations...
deepcave.evaluators.footprint (INFO): Found 3200 configurations...
deepcave.evaluators.footprint (INFO): Found 3400 configurations...
deepcave.evaluators.footprint (INFO): Found 3500 configurations...
deepcave.evaluators.footprint (INFO): Found 3700 configurations...
deepcave.evaluators.footprint (INFO): Found 3800 configurations...
deepcave.evaluators.footprint (INFO): Found 3900 configurations...
deepcave.evaluators.footprint (INFO): Found 4000 configurations...
deepcave.evaluators.footprint (INFO): Added 716 border configs and 726 random configs.
deepcave.evaluators.footprint (INFO): Total configurations: 4002.
deepcave.evaluators.footprint (INFO): Getting MDS data...
deepcave.evaluators.footprint (INFO): Training on ob

In [31]:
data_path = "deepcave_logs/dqn_cc_acrobot"
img = get_importances(Path(data_path) / "run_1", algorithm)
img[0].show()

309.44531 490.890625
490.7421875 490.890625
146.78906 490.890625
342.64844 490.890625
455.539062 490.890625
464.945312 490.890625
403.75781 490.890625
0.0 490.890625
0.0 490.890625
0.0 490.890625
342.46094 490.890625
350.08594 490.890625
420.88281 490.890625
393.390625 490.890625
468.09375 490.890625
402.125 490.890625
471.398438 490.890625
0.0 490.890625
330.22656 490.890625
263.96875 490.890625
0.0 490.890625
287.03906 490.890625
490.6875 490.890625
374.35156 490.890625
408.94531 490.890625
254.71094 490.890625
322.50781 490.890625
370.00781 490.890625
348.8125 490.890625
249.02344 490.890625
475.46875 490.890625
399.21094 490.890625
0.0 490.890625
335.60938 490.890625
490.6796875 490.890625
362.125 490.890625
317.80469 490.890625
473.625 490.890625
391.74219 490.890625
0.0 490.890625
490.640625 490.890625
246.07031 490.890625
438.234375 490.890625
0.25780000000003156 490.890625
0.0 490.890625
456.054688 490.890625
408.265625 490.890625
0.0 490.890625
490.640625 490.890625
362.09375 

RuntimeError: Weights have to be strictly positive.